# Iceberg Local Compaction MVP (v3)

This notebook demonstrates the end-to-end functionality of PyIceberg's new `.replace()` transaction API for performing maintenance tasks like compaction.

It implements the `REPLACE` data operation, proving that the underlying dataset has been logically restructured rather than explicitly overwritten.

In [1]:
import os
import sys
# Ensure the local source codebase is loaded instead of the pip-installed package!
sys.path.insert(0, os.path.abspath('..'))

import random
import pyarrow as pa
from pyiceberg.schema import Schema
from pyiceberg.types import LongType, NestedField, StringType
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import IdentityTransform
from pyiceberg.catalog.sql import SqlCatalog

# Setup standard logging
import logging
logging.getLogger("pyiceberg").setLevel(logging.INFO)

## 1. Environment and Catalog Setup
We'll start by creating a local SQLite catalog and defining a partitioned schema for our table.

In [2]:
# Create local warehouse directory
warehouse_path = "/tmp/iceberg_compaction_v3"
os.makedirs(warehouse_path, exist_ok=True)

# Initialize local SQLite catalog
catalog = SqlCatalog("local_catalog", **{
    "uri": f"sqlite:///{warehouse_path}/catalog.db",
    "warehouse": warehouse_path
})

# Clean up existing namespace if needed
try:
    catalog.drop_table("default.compaction_demo")
except:
    pass

# Create a standard namespace
catalog.create_namespace_if_not_exists("default")

## 2. Defining Schema & Creating the Table
We'll use a schema containing an ID, category, and value, partitioning by category.

In [3]:
schema = Schema(
    NestedField(1, "id", LongType()),
    NestedField(2, "category", StringType()),
    NestedField(3, "value", LongType()),
)
spec = PartitionSpec(PartitionField(source_id=2, field_id=1000, transform=IdentityTransform(), name="category"))

table = catalog.create_table(
    "default.compaction_demo",
    schema=schema,
    partition_spec=spec,
)
print(f"Created Table: {table.name()}")

Created Table: ('default', 'compaction_demo')


## 3. Populating Data (The Append Phase)
To create a fragmented table that realistically needs compaction, we'll append several very small DataFrames. Each operation will produce a fast-append snapshot.

In [4]:
categories = ["cat1", "cat2"]
for i in range(10):
    table.append(
        pa.table({
            "id": list(range(i * 10, (i + 1) * 10)),
            "category": [categories[i % 2]] * 10,
            "value": [random.randint(1, 100) for _ in range(10)],
        })
    )

/Users/jaredyu/Desktop/open_source/iceberg-python/pyiceberg/avro/file.py:175: UserWarning: Falling back to pure Python Avro decoder, missing Cython implementation
  self.decoder = new_decoder(f.read())
/Users/jaredyu/Desktop/open_source/iceberg-python/pyiceberg/avro/file.py:204: UserWarning: Falling back to pure Python Avro decoder, missing Cython implementation
  self.block = Block(reader=self.reader, block_records=block_records, block_decoder=new_decoder(block_bytes))


In [5]:
# Let's verify our uncompacted state
initial_data_files = list(table.scan().plan_files())
print(f"Number of DataFiles in table currently: {len(initial_data_files)}")

initial_df = table.scan().to_arrow()
print(f"Total Number of Rows in logic table: {initial_df.num_rows}")

Number of DataFiles in table currently: 10
Total Number of Rows in logic table: 100


## 4. Understanding `.replace()` vs `.overwrite()`

If we were to use the standard `.overwrite()` API, Iceberg would generate an `OVERWRITE` operation. This implies to downstream consumers that the logical records of the table have fundamentally changed.

The `.replace()` transaction is explicitly built for maintenance work. It produces a `REPLACE` snapshot, proving mathematically to the engine that data is strictly being aggregated, untouched at the record level.

### Call Stack Documentation
Below is the exact execution flow of the highly rigorous `.replace()` API invoked by `table.maintenance.compact()`:

1. **`Table.maintenance.compact()`** (in `pyiceberg/table/maintenance.py`)
    * Scans the table to load all physical `FileScanTask`s into memory.
    * Fetches the actual table payload as an Arrow table: `arrow_table = table.scan().to_arrow()`.
    * Collects the list of explicit `DataFile` references previously stored: `files_to_delete = [task.file for task in table.scan().plan_files()]`.
    * Opens a high-level transaction: `with table.transaction() as txn: txn.replace(..., files_to_delete=...)`.
    
2. **`Transaction.replace()`** (in `pyiceberg/table/__init__.py`)
    * Validates the generic layout string (e.g. "snapshot-type": "replace").
    * Verifies PyArrow is capable of bridging the schema boundaries via `_check_pyarrow_schema_compatible()`.
    * Spawns an explicit builder via the `UpdateSnapshot.replace()` producer factory.
    
3. **`UpdateSnapshot.replace()` -> `_RewriteFiles`** (in `pyiceberg/table/update/snapshot.py`)
    * The producer factory deliberately initializes the `_RewriteFiles` class with `operation=Operation.REPLACE`.
    * This defines the behavior of how the `ManifestList` is physically composed upon writing.
    * The producer logs the previously gathered `files_to_delete` as purely logical soft-deletes: `replace_snapshot.delete_data_file(file_to_delete)`.
    * Iceberg maps the newly compacted Arrow data directly to grouped, compacted Parquet files using binpacking mechanisms and adds them to this operation: `replace_snapshot.append_data_file(data_file)`.

4. **`_RewriteFiles._commit()`**
    * Computes statistical totals mapping the exact sizes, rows, and counts.
    * Safely stages and applies the commit metadata. Because the `operation` traces identically back to `REPLACE`, downstream CDC (change-data-capture) consumers will safely ignore these rewritten metrics.


In [6]:
# Trigger the new compaction implementation
table.maintenance.compact()

print("Compaction complete.")

Compaction complete.


## 5. Verifying the Results
Now we'll prove that the `REPLACE` operation correctly consolidated everything into just two partition files exactly matching `cat1` and `cat2`, without dropping or duplicating rows.

In [7]:
compact_data_files = list(table.scan().plan_files())
print(f"\nNumber of DataFiles in table currently: {len(compact_data_files)}")
assert len(compact_data_files) == 2, "Table should be compacted into precisely 2 partition files."

final_df = table.scan().to_arrow()
print(f"Total Number of Rows in logic table: {final_df.num_rows}")
assert final_df.num_rows == 100, "No logical records should have been lost or duplicated."

print("\n\nSUCCESS! The table was perfectly compacted via the new replace API.")


Number of DataFiles in table currently: 2
Total Number of Rows in logic table: 100


SUCCESS! The table was perfectly compacted via the new replace API.


### Validating Snapshot History Metadata
Finally, we can view the table's snapshot history. Notice the underlying operation type for the most recent summary is exactly `replace`.

In [8]:
table.refresh()
for snapshot in table.metadata.snapshots:
    print(f"Snapshot ID: {snapshot.snapshot_id} | Operation: {snapshot.summary.operation.value}")
    print(f"Properties: {snapshot.summary.additional_properties}")
    print("----")

Snapshot ID: 5469532989379225585 | Operation: append
Properties: {'added-files-size': '1380', 'added-data-files': '1', 'added-records': '10', 'changed-partition-count': '1', 'total-data-files': '1', 'total-delete-files': '0', 'total-records': '10', 'total-files-size': '1380', 'total-position-deletes': '0', 'total-equality-deletes': '0'}
----
Snapshot ID: 9138048806941630342 | Operation: append
Properties: {'added-files-size': '1380', 'added-data-files': '1', 'added-records': '10', 'changed-partition-count': '1', 'total-data-files': '2', 'total-delete-files': '0', 'total-records': '20', 'total-files-size': '2760', 'total-position-deletes': '0', 'total-equality-deletes': '0'}
----
Snapshot ID: 8915354813471332672 | Operation: append
Properties: {'added-files-size': '1380', 'added-data-files': '1', 'added-records': '10', 'changed-partition-count': '1', 'total-data-files': '3', 'total-delete-files': '0', 'total-records': '30', 'total-files-size': '4140', 'total-position-deletes': '0', 'tot